# Preprocessing: Xenium Prime 5K human lymph node

This notebook downloads and prepares the single-section example dataset: a 10x Genomics Xenium Prime 5K run on an FFPE human reactive lymph node. It saves the following file: `~/spectrast_data/xenium_lymph_node_5k.h5ad`, which contains the following: 

- `X`: log-normalised expression (each cell scaled to 10,000 counts, then `log1p`);
- `layers["counts"]`: the raw transcript counts of the cells and genes that pass quality control;
- `obsm["spatial"]`: cell centroids in µm;
- `obs`: the per-cell columns of 10x's `cells.parquet` (transcript counts, cell and nucleus area, segmentation method, ...), plus `cell_type` (10x's labels) and `sample`.

**Requirements.** `scanpy`, `anndata`, `pandas`, `pyarrow` (reads `cells.parquet`) and `remotezip` (reads single files out of a remote zip; `pip install remotezip`). The download is about 240 MB.

In [ ]:
import os, time
import numpy as np, pandas as pd, scanpy as sc, anndata as ad
import requests
from remotezip import RemoteZip

DATA_DIR = os.path.expanduser("~/spectrast_data")
RAW = os.path.join(DATA_DIR, "xenium_lymph_node_5k_raw")       # downloaded files land here
OUT = os.path.join(DATA_DIR, "xenium_lymph_node_5k.h5ad")      # the file the downstream tutorial uses
TMP = os.path.join(DATA_DIR, "xenium_lymph_node_5k.tmp.h5ad")  # written first, then moved to OUT
os.makedirs(RAW, exist_ok=True)

def short(path):
    """Print paths relative to the home directory."""
    return path.replace(os.path.expanduser("~"), "~", 1)

## 1. Source

The dataset is *Preview Data: FFPE Human Lymph Node with 5K Pan Tissue and Pathways Panel* from 10x Genomics, released under a CC BY 4.0 licence (dataset page: https://www.10xgenomics.com/datasets/preview-data-xenium-prime-gene-expression). 10x notes that this preview was generated with development versions of the panel, assay user guide and analysis software.

The Xenium output bundle (`outs.zip`) holds 24 files totalling 22.6 GB, most of it the decoded transcripts and the morphology images. We need only two files (217 MB): the cell × gene count matrix (`cell_feature_matrix.h5`) and the per-cell table with centroids (`cells.parquet`). A zip archive keeps an index of its contents at the end of the file, so `remotezip` reads that index and then fetches just the byte ranges of the two files we want with HTTP range requests. The cell-type labels are a separate CSV on the 10x site.

In [2]:
BUNDLE = ("https://s3-us-west-2.amazonaws.com/10x.files/samples/xenium/3.0.0/"
          "Xenium_Prime_Human_Lymph_Node_Reactive_FFPE/"
          "Xenium_Prime_Human_Lymph_Node_Reactive_FFPE_outs.zip")
CELL_TYPES = ("https://cf.10xgenomics.com/samples/xenium/3.0.0/"
              "Xenium_Prime_Human_Lymph_Node_Reactive_FFPE/"
              "Xenium_Prime_Human_Lymph_Node_Reactive_FFPE_cell_types.csv")
MEMBERS = ["cell_feature_matrix.h5", "cells.parquet"]

# what is in the bundle, and how much of it we fetch (sizes as stored in the zip)
with RemoteZip(BUNDLE, timeout=120) as z:
    inventory = pd.DataFrame([(i.filename, i.compress_size / 1e6) for i in z.infolist() if not i.is_dir()],
                             columns=["file", "MB"]).sort_values("MB", ascending=False)
wanted = inventory["file"].str.split("/").str[-1].isin(MEMBERS)
print(f"bundle: {len(inventory)} files, {inventory.MB.sum() / 1e3:.1f} GB")
print(f"we fetch {wanted.sum()} of them, {inventory.MB[wanted].sum():.0f} MB")
inventory.head(8).round(1)

bundle: 24 files, 22.6 GB
we fetch 2 of them, 217 MB


,file,MB
15,morphology.ome.tif,8625.7
19,transcripts.zarr.zip,4933.3
18,transcripts.parquet,2979.6
14,morphology_focus/morphology_focus_0003.ome.tif,919.0
12,morphology_focus/morphology_focus_0001.ome.tif,898.5
11,morphology_focus/morphology_focus_0000.ome.tif,822.6
13,morphology_focus/morphology_focus_0002.ome.tif,821.4
22,cell_feature_matrix.tar.gz,701.9


## 2. Download

Both downloads write to a `.part` file and rename it  when it is complete, so a file that exists is never truncated, and a re-run skips files already on disk. Each request has a timeout so that a stalled connection fails instead of hanging. This is a little overkill, but best practice since these files can sometimes take a while over slow networks. 

In [3]:
def fetch_members(url, outdir, members, timeout=120, tries=5):
    """Copy `members` out of the remote zip at `url` into `outdir` by HTTP range requests."""
    with RemoteZip(url, timeout=timeout) as z:
        names = {n.split("/")[-1]: n for n in z.namelist()}
        for m in members:
            dst = os.path.join(outdir, m)
            if os.path.exists(dst) and os.path.getsize(dst) > 0:
                print(f"{m}: already present ({os.path.getsize(dst) / 1e6:.1f} MB)")
                continue
            t0 = time.perf_counter()
            for attempt in range(1, tries + 1):
                try:
                    with z.open(names[m]) as src, open(dst + ".part", "wb") as f:
                        f.write(src.read())
                    break
                except Exception as exc:                      # a timeout or a dropped connection
                    print(f"{m}: attempt {attempt} failed ({type(exc).__name__}: {exc})")
                    if attempt == tries:
                        raise
            os.replace(dst + ".part", dst)
            print(f"{m}: {os.path.getsize(dst) / 1e6:.1f} MB in {time.perf_counter() - t0:.0f} s")


def fetch_url(url, dst, timeout=120):
    """Download one file, via `<dst>.part`."""
    if os.path.exists(dst) and os.path.getsize(dst) > 0:
        print(f"{os.path.basename(dst)}: already present ({os.path.getsize(dst) / 1e6:.1f} MB)")
        return
    t0 = time.perf_counter()
    with requests.get(url, stream=True, timeout=timeout) as r, open(dst + ".part", "wb") as f:
        r.raise_for_status()
        for chunk in r.iter_content(16 * 2**20):
            f.write(chunk)
    os.replace(dst + ".part", dst)
    print(f"{os.path.basename(dst)}: {os.path.getsize(dst) / 1e6:.1f} MB in {time.perf_counter() - t0:.0f} s")


t0 = time.perf_counter()
fetch_members(BUNDLE, RAW, MEMBERS)
fetch_url(CELL_TYPES, os.path.join(RAW, "cell_types.csv"))
print(f"total: {time.perf_counter() - t0:.0f} s")

cell_feature_matrix.h5: 206.9 MB in 55 s


cells.parquet: 13.2 MB in 2 s


cell_types.csv: 21.4 MB in 2 s
total: 59 s


## 3. Assemble the AnnData

Besides the 4,624 genes, `cell_feature_matrix.h5` stores 6,470 control probes and codewords; we keep only the Gene Expression features. `cells.parquet` is aligned to the matrix's cell order by `cell_id`. Its `x_centroid` and `y_centroid` columns (µm) become `obsm["spatial"]`, and its other columns go into `obs`.

In [ ]:
import h5py
h5 = os.path.join(RAW, "cell_feature_matrix.h5")
with h5py.File(h5, "r") as f:                            # feature types stored in the matrix
    print(pd.Series(f["matrix/features/feature_type"][:].astype(str)).value_counts().to_string(), "\n")
a = sc.read_10x_h5(h5, gex_only=True)                  # Gene Expression features only
a.var_names_make_unique()

cells_tbl = pd.read_parquet(os.path.join(RAW, "cells.parquet")).set_index("cell_id").reindex(a.obs_names)
print(f"matrix cells without a row in cells.parquet: {int(cells_tbl['x_centroid'].isna().sum())}")
a.obsm["spatial"] = cells_tbl[["x_centroid", "y_centroid"]].to_numpy(float)
for c in cells_tbl.columns.drop(["x_centroid", "y_centroid"]):
    a.obs[c] = cells_tbl[c].to_numpy()
a.obs_names = a.obs_names.astype(str)
print(a)

Gene Expression              4624
Deprecated Codeword          3291
Unassigned Codeword          2509
Negative Control Codeword     609
Negative Control Probe         40
Genomic Control                21 



matrix cells without a row in cells.parquet: 0


AnnData object with n_obs × n_vars = 708983 × 4624
    obs: 'transcript_counts', 'control_probe_counts', 'genomic_control_counts', 'control_codeword_counts', 'unassigned_codeword_counts', 'deprecated_codeword_counts', 'total_counts', 'cell_area', 'nucleus_area', 'nucleus_count', 'segmentation_method'
    var: 'gene_ids', 'feature_types', 'genome'
    obsm: 'spatial'

## 4. Cell-type labels

10x provides a CSV of putative cell-type labels (cell_id, group), transferred from public scRNA-seq references on CZ CELLxGENE. These labels are intended for exploration and are not guaranteed to be publication-quality. One known error is a putative MARCO+ macrophage cluster labelled as endothelial.

The labels also do not distinguish germinal-centre from mantle-zone B cells: the available B-lineage categories are B Cell, Naive B Cell, Memory B Cell, and Plasma Cell. All 708,983 cells matched a CSV entry, spanning 28 labels; unmatched cells would be assigned "unlabelled".

In [5]:
T = pd.read_csv(os.path.join(RAW, "cell_types.csv"))
print("columns:", list(T.columns), f"| rows: {len(T):,}")
lab = T.set_index("cell_id")["group"].astype(str).reindex(a.obs_names)
a.obs["cell_type"] = pd.Categorical(lab.fillna("unlabelled").to_numpy())
a.obs["sample"] = "xenium_lymph_node_5k"
a.uns["platform"] = "10x Xenium Prime 5K (Human 5K Pan Tissue & Pathways development panel)"
a.uns["labels"] = ("10x Genomics cell_types.csv: label transfer against CZ CELLxGENE "
                   "references, released with the dataset")

n_lab = int(lab.notna().sum())
print(f"labelled: {n_lab:,} of {a.n_obs:,} cells ({n_lab / a.n_obs:.2%}); unlabelled: {a.n_obs - n_lab:,}")
print(f"label rows naming no cell in the matrix: {len(set(T['cell_id']) - set(a.obs_names)):,}")
print(f"{a.obs['cell_type'].nunique()} distinct values of cell_type")
a.obs["cell_type"].value_counts().rename("cells").to_frame()

columns: ['cell_id', 'group'] | rows: 708,983
labelled: 708,983 of 708,983 cells (100.00%); unlabelled: 0


label rows naming no cell in the matrix: 0
28 distinct values of cell_type


,cells
cell_type,
Endothelial Cell,120608
Memory B Cell,90051
Naive Thymus Derived CD4+ αβ T Cell,62350
Stromal Cell,58591
Regulatory T Cell,47902
Macrophage,39797
B Cell,36453
Effector CD4+ αβ T Cell,31823
Innate Lymphoid Cell,31405


## 5. Quality control

Five filters, each standard for Xenium data.

**Transcripts.** Xenium gives every decoded transcript a quality score (Q). The count matrix already holds only the transcripts with Q ≥ 20 that fall inside a cell, and the control probes and codewords sit in separate features, which we dropped above. The check below confirms this from 10x's own run metrics: the number of Q20 transcripts times the fraction assigned to cells equals the matrix total. A stricter cutoff would mean rebuilding the matrix from `transcripts.parquet` (3.0 GB), which carries each transcript's Q score.

**Segmentation.** This run used the Xenium multimodal cell segmentation stain. Each cell is outlined by the boundary stain (ATP1A1, CD45 and E-cadherin), by the interior stain (18S rRNA) or, where neither resolves it, by expanding its nucleus by 5 µm. Nucleus-expansion cells have only an approximate outline, and here they are also low in signal, so we remove them.

**Low-quality cells.** We keep cells with at least 30 transcripts and at least 10 detected genes.

**Size.** We remove cells below the 1st or above the 99th percentile of cell area: the smallest are often fragments and the largest merged cells.

**Genes.** We remove genes detected in fewer than 10 of the cells that remain.

The size percentiles are computed over all cells, and the cell filters are applied together. The table shows how many cells each filter removes, and how many it alone removes.


In [ ]:
fetch_members(BUNDLE, RAW, ["metrics_summary.csv"])           # 10x's run metrics, 2 KB
m = pd.read_csv(os.path.join(RAW, "metrics_summary.csv")).iloc[0]
q20, assigned = m["total_high_quality_decoded_transcripts"], m["fraction_transcripts_assigned"]
in_matrix = float(a.X.sum())
print(f"Q20 transcripts: {q20:,.0f}; fraction assigned to cells: {assigned:.4f}; expected in cells: {q20 * assigned:,.0f}")
print(f"counts in the matrix: {in_matrix:,.0f} (ratio {in_matrix / (q20 * assigned):.5f})")
print(f"10x's estimate of false-positive transcripts per cell: "
      f"{m['estimated_number_of_false_positive_transcripts_per_cell']:.2f}")


In [ ]:
MIN_COUNTS, MIN_GENES, AREA_PCT, MIN_CELLS = 30, 10, (1, 99), 10

seg = a.obs["segmentation_method"].astype(str)
counts = np.asarray(a.X.sum(axis=1)).ravel()
genes = np.asarray((a.X > 0).sum(axis=1)).ravel()
area = a.obs["cell_area"].to_numpy(float)
lo, hi = np.percentile(area, AREA_PCT)

per_method = pd.DataFrame({"cells": seg.value_counts()})
per_method["median transcripts"] = [np.median(counts[(seg == s).to_numpy()]) for s in per_method.index]
per_method["median area (µm²)"] = [np.median(area[(seg == s).to_numpy()]) for s in per_method.index]
print(per_method.round(0).to_string(), "\n")

fail = pd.DataFrame({
    "segmented by nucleus expansion": seg.str.contains("nucleus expansion").to_numpy(),
    f"fewer than {MIN_COUNTS} transcripts": counts < MIN_COUNTS,
    f"fewer than {MIN_GENES} genes": genes < MIN_GENES,
    f"area outside {lo:.1f}-{hi:.1f} µm²": (area < lo) | (area > hi),
}, index=a.obs_names)
keep = ~fail.any(axis=1).to_numpy()
only = fail[fail.sum(axis=1) == 1]
table = pd.DataFrame({"cells removed": fail.sum(), "removed by this filter alone": only.sum()})
table.loc["any filter"] = [int((~keep).sum()), ""]

a = a[keep].copy()
n_genes = a.n_vars
sc.pp.filter_genes(a, min_cells=MIN_CELLS)
print(f"kept {keep.sum():,} of {len(keep):,} cells ({keep.mean():.1%}); "
      f"kept {a.n_vars:,} of {n_genes:,} genes (detected in at least {MIN_CELLS} cells)")
table


## 6. Normalisation

We keep the raw counts of the retained cells and genes in `layers["counts"]`, then scale every cell to 10,000 counts and take `log1p`. Scaling removes differences in how many transcripts each cell yielded, and the log keeps a few very high counts from dominating a gene's spatial pattern. Spectrast standardises each gene before measuring its organization, so the target sum itself does not matter.


In [ ]:
a.layers["counts"] = a.X.copy()
sc.pp.normalize_total(a, target_sum=1e4)
sc.pp.log1p(a)
a


## 7. Summary

The section has 708,983 cells and 4,624 genes over about 8.3 × 7.2 mm, with a median of 255 counts per cell (the dataset page also reports a median of 255 transcripts per cell). The median distance from each cell centroid to its nearest neighbor, 5.0 µm, is on the scale of a single cell, a quick check that the coordinates are in micrometres.

In [6]:
from scipy.spatial import cKDTree
xy = a.obsm["spatial"]
ext = np.ptp(xy, axis=0)
counts = np.asarray(a.layers["counts"].sum(axis=1)).ravel()
nn = cKDTree(xy).query(xy, k=2)[0][:, 1]
print(f"{a.n_obs:,} cells x {a.n_vars:,} genes; {a.X.nnz:,} non-zeros")
print(f"counts per cell: median {np.median(counts):.0f}, range {counts.min():.0f}-{counts.max():.0f}")
print(f"spatial extent: {ext[0]:,.0f} x {ext[1]:,.0f} µm "
      f"(x {xy[:, 0].min():.0f}-{xy[:, 0].max():.0f}, y {xy[:, 1].min():.0f}-{xy[:, 1].max():.0f})")
print(f"median nearest-neighbour distance: {np.median(nn):.1f} µm")

708,983 cells x 4,624 genes; X is csr_matrix, float32, 157,025,660 non-zeros
counts per cell: median 255, range 0-2769
spatial extent: 8,302 x 7,211 µm (x 32-8334, y 33-7243)
median nearest-neighbour distance: 5.0 µm


## 8. Write the file

We write to a temporary file and move it into place with `os.replace`, so a half-written file never sits at the path the analysis tutorial reads.


In [ ]:
a.write_h5ad(TMP)
os.replace(TMP, OUT)
print("wrote", short(OUT), f"({os.path.getsize(OUT) / 1e9:.2f} GB)")
